\setcounter{secnumdepth}{0}

# Methylatie bestand #

CCLE_RRBS_TSS1kb_20181022.txt - bevat DNA-methylatiegegevens gemeten rond de Transcription Start Site (TSS, ±1 kb). Dit gebied omvat zowel promoterregio's als het eerste exon, die belangrijk zijn voor genregulatie.

### Stap 1. Inlezen en verkennen van de methylatie-data ###

Doel: de methylatie-data van CCLE inlezen en een eerste verkenning doen. 

We bekijken het aantal rijen/kolommen, de eerste en laatste kolomnamen, datatypes en missing values.

Omdat we alleen methylatie van longkankercellijnen willen analyseren selecteren we de kolommen waarvan de naam _LUNG bevat. Het resultaat is een dataframe met:
- Metadata (locuc_id, CpG_sites_Hg19, avg_coverage
- Akkeen de cellijnkolommen die longkanker representeren

In [1]:
import pandas as pd
import numpy as np

# aanmaken variabele met de bestandpad
methylatie_file = "../data/raw/CCLE_RRBS_TSS1kb_20181022.txt"

# inlezen
methylatie_df = pd.read_csv(methylatie_file, sep='\t', low_memory=False) # txt, tab seperated

# verkenning
print("Aantal rijen en kolommen:", methylatie_df.shape) # globale info

# eerste en laatste 5 kolommen bekijken
print("Eerste 5 kolommen:", methylatie_df.columns[:5].tolist())
print("Laatste 5 kolommen:", methylatie_df.columns[-5:].tolist())

# datatypes en missende waarden per kolom bekijken
kolom_info_methylatie = pd.DataFrame({ # aanmaken dataframe (met 3 kolommen)
    'kolom': methylatie_df.columns,
    'dtype': [methylatie_df[col].dtype for col in methylatie_df.columns],
    'n_missing': [methylatie_df[col].isna().sum() for col in methylatie_df.columns] # bepaald totaal aantal missende waarden voor elke kolom
})

# eerste 10 en laatste 10 kolommen tonen als voorbeeld
print(kolom_info_methylatie.head(5)) # eerste 10 kolommen
print(kolom_info_methylatie.tail(5)) # laatste 10 kolommen

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.
Aantal rijen en kolommen: (21338, 846)
Eerste 5 kolommen: ['locus_id', 'CpG_sites_hg19', 'avg_coverage', 'DMS53_LUNG', 'SW1116_LARGE_INTESTINE']
Laatste 5 kolommen: ['UO31_KIDNEY', 'SF268_CENTRAL_NERVOUS_SYSTEM', 'SF539_CENTRAL_NERVOUS_SYSTEM', 'SNB75_CENTRAL_NERVOUS_SYSTEM', 'HOP92_LUNG']
                    kolom   dtype  n_missing
0                locus_id  object          0
1          CpG_sites_hg19  object          1
2            avg_coverage  object          0
3              DMS53_LUNG  object          0
4  SW1116_LARGE_INTESTINE  object          0
                            kolom   dtype  n_missing
841                   UO31_KIDNEY  object          0
842  SF268_CENTRAL_NERVOUS_SYSTEM  object          0
843  SF539_CENTRAL_NERVOUS_SYSTEM  object          0
844  SNB75_CENTRAL_NERVOUS_SYSTEM  object          0
845                 

Waarneming:

- De dataset bevat 21338 rijen die CpG-loci representeren en 846 kolommen.
- De eerste 3 kolommen zijn metadata:
  - locus_id: unieke identifier van de CpG-locus  
  - CpG_sites_hg19:genoompositie van de locus  
  - avg_coverage: gemiddelde meetdiepte, een maat voor de   betrouwbaarheid van de meting  
- De overige 843 kolommen zijn de cellijnen.   
- Wat me opvalt is dat dit bestand geen ACH- of ModelID bevat. In plaats daarvan worden cellijnen geidentificeerd op basis van hun naam, waarbij de kolomtitel zowel de cellijnnaam als het weefseltype bevat (bijvoorbeeled DMS53_LUNG).

Om uiteindelijk 3 datasets samen te voegen zal een mapping moeten worden uitgevoerd van deze kolomnamen naar de bijbehorende ACH_ID.

### Stap 2. Filteren op longkankercellijnen ###

Doel: Alleen de kolommen behouden die longkankercellijnen representeren (_LUNG). Zo wordt er een subset aangemaakt van de relevante data.



In [2]:
# filter kolommen die longkankercellijnen representeren
lung_columns = [col for col in methylatie_df.columns if 'lung' in col.lower()]

# maak dataframe met longkankercellijnen + metadata
lung_methylatie_df = methylatie_df[['locus_id', 'CpG_sites_hg19', 'avg_coverage'] + lung_columns]

print(f"Aantal longkankercellijnen: {len(lung_columns)}")
print("Vorm van longkankermethylatie matrix:", lung_methylatie_df.shape)
print(lung_methylatie_df.head(5))
      

Aantal longkankercellijnen: 153
Vorm van longkankermethylatie matrix: (21338, 156)
                    locus_id  \
0  SGIP1_1_66998638_66999638   
1  SGIP1_1_66998251_66999251   
2  AZIN2_1_33545713_33546713   
3  AZIN2_1_33546778_33547778   
4  AGBL4_1_50489626_50490626   

                                      CpG_sites_hg19 avg_coverage DMS53_LUNG  \
0  1:66998970;1:66998973;1:66998993;1:66999404;1:...        25.00    0.00000   
1                   1:66998970;1:66998973;1:66998993         8.27    0.00000   
2  1:33546151;1:33546209;1:33546210;1:33546385;1:...       326.58    0.00729   
3  1:33546783;1:33546788;1:33546795;1:33546797;1:...       480.54    0.22276   
4  1:50489632;1:50489641;1:50489671;1:50489677;1:...       263.36    0.00000   

  NCIH1184_LUNG NCIH2227_LUNG RERFLCAD2_LUNG NCIH2347_LUNG NCIH2087_LUNG  \
0       0.11864       0.04545            NaN       0.25000       0.86441   
1       0.22579           NaN            NaN           NaN       0.97871   
2       0.22553

### Stap 3. Quality Control (QC) ###

Er worden een aantal stappen uitgevoerd om de dataset betrouwbaar, schoon en geschikt te maken voor analyse.

(...eventueel stappen benoemen...)

#### Coverage Filtering: rijen met lage avg_coverage verwijderen ####


Doel:  
Bij methylatie-metingen varieert de meetdiepte (`avg_coverage`) per CpG-locus. Lage coverage kan leiden tot onbetrouwbare waarden en ruis in downstream analyses. Daarom worden loci met lage gemiddelde coverage weggefilterd, zodat alleen loci met voldoende betrouwbare metingen overblijven.

Aanpak: 
De kolom `avg_coverage` wordt vanuit het tekstbestand ingelezen als string, terwijl het in werkelijkheid een getal is. Om er mee te kunnen rekenen of filteren, moet de kolom worden omgezet naar een numeriek datatype. 

Sommige loci kunnen een ontbrekende of ongeldige coveragewaarde (NaN) hebben. Voor deze loci is het onmogelijk om te betrouwbaarheid van de meting te beoordelen. Deze rijen leveren geen bruikbare informatie op en worden daarom verwijderd. 

Vervolgens worden alleen loci behouden met een avg_coverage ≥ 10.Loci met lage coverage bevatten ruis en onbetrouwbare methylatiewaarden. Door 

Het resultaat is een subset van CpG-loci die kwalitatief sterk genoeg is voor downstream analyse.

In [3]:
# maak kopie 
lung_methylatie_qc = lung_methylatie_df.copy()

# converteer avg_coverage naar numeriek
lung_methylatie_qc['avg_coverage'] = pd.to_numeric(lung_methylatie_qc['avg_coverage'], errors ='coerce')

# verwijder rijen met NaN in avg_coverage
lung_methylatie_qc = lung_methylatie_qc.dropna(subset=['avg_coverage'])

# filter loci met avg_coverage >=10
lung_methylation_qc = lung_methylatie_qc[lung_methylatie_qc['avg_coverage']>=10]

# check hoe veel loci overblijven na coverage filtering
print(f"Aantal loci voor filtering op avg coverage: {len(lung_methylatie_df)}")
print(f"Aantal loci na coverage filtering: {len(lung_methylation_qc)}")


Aantal loci voor filtering op avg coverage: 21338
Aantal loci na coverage filtering: 20898


#### Missingness: rijen met teveel ontbrekende waarden verwijderen ####


Doel: 
Verwijderen van CpG-loci die te veel ontbrekende methylatiewaarden hebben over longkankercellijnen. Dit vermindert ruis en zorgt dat downstream analyses betrouwbaarder zijn.

Aanpak:   
Voor elke locus wordt het percentage ontbrekende waarden berekend. Loci met meer dan 10% missende waarden worden verwijderd. Deze drempel is relatief streng en zorgt ervoor dat de overgebleven loci bijna overal gemeten zijn, wat de kwaliteit van de dataset verhoogd.

In [4]:
# maak kopie
lung_methylation_qc = lung_methylation_qc.copy()

# selecteer alleen methylatiekolommen (dus alle cellijnen)
methylation_cols = [col for col in lung_methylation_qc.columns 
                    if col not in ['locus_id', 'CpG_sites_hg19', 'avg_coverage']]

# converteer methylatiekolommen naar numeriek
lung_methylation_qc[methylation_cols] = lung_methylation_qc[methylation_cols].apply(pd.to_numeric, errors='coerce')

# bereken missingness per locus (rij)
lung_methylation_qc['missing_fraction'] = lung_methylation_qc[methylation_cols].isna().mean(axis=1)

# instellen van de missingness drempel
missingness_threshold = 0.10

# filter loci met teveel missende waarden
before = len(lung_methylation_qc)
lung_methylation_qc = lung_methylation_qc[lung_methylation_qc['missing_fraction'] <= missingness_threshold]
after = len(lung_methylation_qc)

# feedback
print(f"Loci vóór missingness filtering: {before}")
print(f"Loci na missingness filtering: {after}")
print(f"Verwijderd door missingness: {before - after}")


Loci vóór missingness filtering: 20898
Loci na missingness filtering: 18449
Verwijderd door missingness: 2449


/var/folders/hf/q9gpk8mn1l34cwvqff64g18c0000gn/T/ipykernel_21628/1362123296.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  lung_methylation_qc['missing_fraction'] = lung_methylation_qc[methylation_cols].isna().mean(axis=1)


De overgebleven loci zijn betrouwbaar gemeten in het merendeel van de cellijnen en vormen een goede basis voor de volgende QC-stap. Ongeveer 12% van de loci is verwijderd wat betekend dat de dataset aanzienlijk is schoongemaakt van ruis zonder te veel relevante informatie te verliezen.

#### Variance Filtering ####


Doel:  
Het verwijderen van  CpG-loci die weinig variatie laten zien over de longkankercellijnen. 

Loci met bijna constante methylatiewaarden dragen weinig bij aan downstream analyses en kunnen ruis toevoegen. 

Aanpak: 
Eerst wordt de standaarddeviatie (std) per locus over alle longkankercellijnen berekent. Er wordt een drempel ingesteld waarmee loci met std < 0.05 worden verwijderd. Deze drempel is gangbaar in methylatieonderzoek waarbij loci met std ≥ 0.05 voldoende variatie laten zien en als informatief worden beschouwd.

Vervolgens moet er een drempel worden ingesteld waarbij loci met variantie lager dan een gekozen waarde worden verwijderd. 

In [5]:

# maak kopie
lung_methylation_qc = lung_methylation_qc.copy()

# bereken variantie per locus over alle methylatiekolommen
lung_methylation_qc['variance'] = lung_methylation_qc[methylation_cols].var(axis=1)

# stel drempel in voor minimale variantie
variance_threshold = 0.05

# filter loci met variantie >= drempel
before_var = len(lung_methylation_qc)
lung_methylation_qc = lung_methylation_qc[lung_methylation_qc['variance'] >= variance_threshold]
after_var = len(lung_methylation_qc)

# feedback
print(f"Loci vóór variance filtering: {before_var}")
print(f"Loci na variance filtering: {after_var}")
print(f"Verwijderd door lage variantie: {before_var - after_var}")


Loci vóór variance filtering: 18449
Loci na variance filtering: 4569
Verwijderd door lage variantie: 13880


Ongeveer 75% van de loci vertoonde te weinig variatie en zijn verwijderd, wat normaal is bij genome-wide methylatie-data, aangezien veel CpG-loci vrijwel constant of weinig variabel zijn over de cellijnen. 

### Stap 4. Mapping en transponeren van de methylatie-data ###

Doel: Methylatie-data voorbereiden op integratie met de andere datasets (mutaties en responswaarden).

Daarvoor moeten we de cellijnnamen in de methylatie-dataset eerst worden vertaald naar hun bijbehorende DepMap/ACH-IDs, zodat alle bestanden een gedeelde unieke identifier gebruiken. Vervolgens wordt de dataset getransponeerd zodat cellijnen rijen worden en loci kolommen, wat het juiste formaat is voor downstream analyse en modelbouw.

Aanpak: 

- Metadata inladen:
  De DepMap-annotatie bevat de koppeling tussen CCLE-namen (zoals DMS53_LUNG) en de standaard DepMap/ACH-IDs. Deze dataset wordt ingeladen en beperkt tot de kolommen die nodig zijn voor mapping: `CCLE_ID` --> `DepMapID`.
  
- Mapping uitvoeren:
  De methylatiekolommen worden vergeleken met de CCLE-IDs in de metadata. Alleen cellijnen waarvoor een geldige DepMap-ID bestaat, worden behouden. De kolomnamen worden vervangen door ACH-IDs.

  
- Transponeren van de dataset:
  Na de mapping wordt het DataFrame omgezet zodat de ACH-IDs de tijen vormen en de methylatie-loci de kolommen. Dit formaat sluit aan bij de andere datasets en maakt latere merge-stappen eenvoudig.


In [6]:
# inlezen van het metadata-bestand en verkenning
metadata_file = "../data/raw/Cell_lines_annotations_20181226.txt"
metadata_df = pd.read_csv(metadata_file, sep='\t', low_memory=False) # txt, tab seperated

print("Metadata ingeladen.")
print(f"Aantal rijen en kolommen: {metadata_df.shape}") 

# belangrijke kolommen selecteren
important_cols = ['CCLE_ID', 'depMapID', 'Name', 'Site_Primary', 
        'Site_Subtype1', 'Site_Subtype2', 'Site_Subtype3', 'Disease']

# alleen tonen als alle kolommen aanwezig zijn
available_cols = [c for c in important_cols if c in metadata_df.columns]

print("\nBeschikbare belangrijke kolommen:")
print(available_cols)

# metadata reduceren tot de kolommen die we nodig hebben voor mapping
metadata_df = metadata_df[['CCLE_ID', 'depMapID']].dropna()

print(f"\nAantal cellijnen bruikbaar voor mapping: {len(metadata_df)}")

# mapping van kolommen naar ACH_ID
methylation_cols = [col for col in lung_methylation_qc.columns 
                    if col not in ['locus_id','CpG_sites_hg19','avg_coverage']]

# mapping: CCLE_ID -> depMapID
ccle2depmap = dict(zip(metadata_df['CCLE_ID'], metadata_df['depMapID']))

# pas mapping toe en behoud alleen kolommen met bestaande mapping
new_cols = []
valid_cols = []
for col in methylation_cols:
    depmap_id = ccle2depmap.get(col)
    if depmap_id:
        new_cols.append(depmap_id)
        valid_cols.append(col)

# maak nieuwe DataFrame met gemapte kolomnamen
meth_mapped_df = lung_methylation_qc[['locus_id','CpG_sites_hg19','avg_coverage'] + valid_cols].copy()
meth_mapped_df.columns = ['locus_id','CpG_sites_hg19','avg_coverage'] + new_cols

print(f"Aantal cellijnen na mapping: {len(new_cols)}")

# transponeren

meth_T = meth_mapped_df.set_index("locus_id").drop(columns=['CpG_sites_hg19','avg_coverage']).T
meth_T.index.name = 'ModelID' # ACH-IDs

print("\nTransponeren voltooid.")
print("Vorm van de getransponeerde methylatie-matrix:", meth_T.shape)


Metadata ingeladen.
Aantal rijen en kolommen: (1461, 33)

Beschikbare belangrijke kolommen:
['CCLE_ID', 'depMapID', 'Name', 'Site_Primary', 'Site_Subtype1', 'Site_Subtype2', 'Site_Subtype3', 'Disease']

Aantal cellijnen bruikbaar voor mapping: 1457
Aantal cellijnen na mapping: 153

Transponeren voltooid.
Vorm van de getransponeerde methylatie-matrix: (153, 4569)


In [7]:
meth_T.head()

locus_id,AGBL4_1_50489626_50490626,SLC45A1_1_8377144_8378144,TGFBR3_1_92351836_92352836,PRKCZ_1_2004424_2005424,PRDM16_1_2984741_2985741,KAZN-AS1_1_14746469_14747469,CSMD2_1_34631443_34632443,RNF220_1_44888495_44889495,SCP2_1_53391900_53392900,DAB1_1_57888784_57889784,...,RIBC2_22_45808571_45809571,FBLN1_22_45897718_45898718,MIR3619_22_46485923_46486923,MIRLET7BHG_22_46480876_46481876,LINC00899_22_46440748_46441748,LINC00898_22_48027318_48028318,LOC284933_22_48943199_48944199,MIR4535_22_49175106_49176106,IL17REL_22_50451055_50452055,TRABD_22_50627978_50628978
ModelID,,,,,,,,,,,,,,,,,,,,,
ACH-000698,0.00000,0.06154,0.06495,0.90697,0.05687,0.03296,0.08394,0.86905,0.10964,0.00000,...,0.00529,0.02528,0.97458,0.92657,0.00000,0.00000,0.00000,0.00277,0.35996,0.89704
ACH-000523,0.07346,0.04211,0.12579,0.17693,0.02214,0.20000,0.19928,0.23962,0.37374,0.02541,...,0.97040,0.19720,0.95495,0.82109,0.17443,0.35302,0.18813,0.20239,0.82296,0.81493
ACH-000610,0.01246,0.40403,0.06295,0.20561,0.00902,0.85541,0.08861,0.58460,0.15685,0.01953,...,0.75786,0.10829,0.44571,0.44586,0.30959,0.42902,0.44565,0.39904,0.35151,0.45146
ACH-000774,0.00531,0.18182,0.09434,0.84290,0.06143,0.36966,0.12902,0.30636,0.20237,0.02556,...,0.56134,0.06710,0.79487,0.15264,0.60137,0.21231,0.73684,0.43176,0.26277,0.50296
ACH-000875,0.07422,0.15942,0.11582,0.94621,0.15642,0.41465,0.79630,0.02632,0.31252,0.24499,...,0.01204,0.37677,1.00000,0.52817,0.01380,0.48066,0.90804,0.40959,0.31794,0.31934


### Resultaat van de methylatie-matrix ###

Ter controle zijn de eerste rijen van de getransponeerde methylatiematrix weergegeven. De eerste kolom bevat de `ModelID`, waarbij elke rij overeenkomt met één cellijn (ACH-ID). Alle overige kolommen vertegenwoordigen individuele CpG-loci. Deze loci hebben een standaard DepMap-namingconventie `GENE_CHROM_START_END`, wat aangeeft bij welk gen de tile hoort en op welke genomische coordinaren deze is gebaseerd.

De methylatie-matrix heeft daarmee het juiste formaat voor verdere analyses: cellijnen als rijen en methylatiefeatures als kolommen. Deze opgeschoonde en gemapte methylatiematrix wordt later gebruikt in het final/merge-notebook voor deelvraag 1, waar alle datasets worden gecombineer tot één dataset

In [8]:
# sla dataframe op als pickle(houdt types en index exact hetzelfde)
meth_T.to_pickle("meth_T.pkl")
